# TP final · Opción B · Experimentos ML

Este notebook entrena. Lee la tabla `survey_cleaned` que deja `00_eda_cleaning.ipynb`, arma una sola matriz y entrena con el mismo corte. Cada modelo queda como un run en MLflow.

La etiqueta es `music_effects`: `Improve`, `No effect`, `Worsen`. La métrica que decide es el F1 macro de test, el promedio del F1 de las tres clases con el mismo peso.

Reglas de filas, ya cerradas después de la exploración:

- Se descartan las filas sin `music_effects`. La etiqueta no se imputa.
- `3-Tratar BPM` arranca en `conservar`: el `bpm` queda como está en la tabla. `anular_imposibles` pasa a nulo un valor menor a 40 o mayor a 250. El rango no viene de la encuesta: marca tempos imposibles, como 0 o 999999999. La fila se queda.
- La mediana de las numéricas y la moda de las categóricas se calculan solo con el train, dentro del pipeline, después del corte 80/20 estratificado (`random_state=7`).
- `timestamp` no entra: es la hora en que mandó el formulario, no un hábito de escucha.

Con los widgets como vienen, una corrida de arriba hacia abajo entrena los cuatro modelos de la comparación. Quedan como `exp_01`, `exp_02`, `exp_03` y `exp_04`. `02_comparacion.ipynb` no vuelve a entrenar: lee esos runs.

## Widgets

Si corrés el notebook entero sin tocar nada, `4-Como entrenar` queda en `automatico` y se registran los cuatro modelos comparables: `exp_01` logística, `exp_02` random forest, `exp_03` LightGBM y `exp_04` CatBoost. No hace falta lanzarlos uno por uno. En ese modo no entran el nombre, el modelo, el balanceo, `C`, los árboles ni la profundidad. `3-Tratar BPM` queda en `conservar`.

Para un experimento distinto, pasá `4-Como entrenar` a `manual`, ajustá el resto y ejecutá de nuevo desde la celda de la matriz. Ese run lleva otro nombre, así `02_comparacion` sigue leyendo `exp_01` a `exp_04`. Si le ponés a propósito uno de esos nombres, el último run con ese nombre pasa a ser el de la comparación.

| Widget | Default | Qué hace |
|---|---|---|
| `1-Nombre Experimento` | vacío | Solo con `manual`. Vacío arma `manual-<modelo>`. |
| `2-Scores` | `todos` | Entran `anxiety`, `depression`, `insomnia` y `ocd`. Vale en los dos modos. `ninguno` no entra ninguno. Si `ninguno` queda marcado junto con otro, se ignora. |
| `3-Tratar BPM` | `conservar` | Deja el `bpm` como está en la tabla. `anular_imposibles` pasa a nulo un valor menor a 40 o mayor a 250. Vale en los dos modos. |
| `4-Como entrenar` | `automatico` | `automatico` ignora los parámetros de abajo y entrena los cuatro. `manual` entrena solo el modelo elegido. |
| `5-Modelo` | `logistica` | Solo con `manual`. |
| `6-Balanceo` | `como_la_comparacion` | Logística sin peso y árboles balanceados. `ninguno` saca el peso. `balanced` se lo pone también a la logística. |
| `7-C` | `1` | Solo logística, y solo con `manual`. |
| `8-Arboles` | `100` | Random forest, LightGBM y CatBoost, solo con `manual`. |
| `9-Profundidad` | `sin_limite` | Solo con `manual`. En CatBoost, `sin_limite` deja la profundidad que trae la librería (6). |

In [ ]:
%pip install -q lightgbm catboost

In [ ]:
%restart_python

In [ ]:
def sacar(nombre):
    try:
        dbutils.widgets.remove(nombre)
    except Exception:
        pass

def previo(nombre, default):
    try:
        return dbutils.widgets.get(nombre)
    except Exception:
        return default

def uno_de(valor, opciones, default):
    return valor if valor in opciones else default

scores_opciones = ["todos", "ninguno", "anxiety", "depression", "insomnia", "ocd"]
como_opciones = ["automatico", "manual"]
modelo_opciones = ["logistica", "random_forest", "lightgbm", "catboost"]
balanceo_opciones = ["como_la_comparacion", "ninguno", "balanced"]
c_opciones = ["0.01", "0.1", "1", "10"]
arboles_opciones = ["50", "100", "200"]
profundidad_opciones = ["3", "5", "8", "sin_limite"]
bpm_opciones = ["conservar", "anular_imposibles"]

nombre_previo = previo("nombre_experimento", "")
scores_previo = uno_de(previo("scores", "todos"), scores_opciones, "todos")
bpm_previo = uno_de(previo("tratar_bpm", "conservar"), bpm_opciones, "conservar")
como_previo = uno_de(previo("como_entrenar", "automatico"), como_opciones, "automatico")
modelo_previo = uno_de(previo("modelo", "logistica"), modelo_opciones, "logistica")
balanceo_previo = uno_de(previo("balanceo", "como_la_comparacion"), balanceo_opciones, "como_la_comparacion")
c_previo = uno_de(previo("C", "1"), c_opciones, "1")
arboles_previo = uno_de(previo("n_arboles", "100"), arboles_opciones, "100")
profundidad_previo = uno_de(previo("profundidad", "sin_limite"), profundidad_opciones, "sin_limite")

for viejo in (
    "puntajes_salud", "que_entrenar", "nombre_run",
    "nombre_experimento", "scores", "como_entrenar", "modelo",
    "balanceo", "C", "n_arboles", "profundidad", "tratar_bpm", "layout_scores_bpm",
):
    sacar(viejo)
dbutils.widgets.text("nombre_experimento", nombre_previo, "1-Nombre Experimento")
dbutils.widgets.multiselect(
    "scores", scores_previo, scores_opciones, "2-Scores",
)
dbutils.widgets.dropdown(
    "tratar_bpm", bpm_previo, bpm_opciones, "3-Tratar BPM"
)
dbutils.widgets.dropdown(
    "como_entrenar", como_previo, como_opciones, "4-Como entrenar"
)
dbutils.widgets.dropdown(
    "modelo", modelo_previo, modelo_opciones, "5-Modelo"
)
dbutils.widgets.dropdown(
    "balanceo", balanceo_previo, balanceo_opciones, "6-Balanceo"
)
dbutils.widgets.dropdown("C", c_previo, c_opciones, "7-C")
dbutils.widgets.dropdown("n_arboles", arboles_previo, arboles_opciones, "8-Arboles")
dbutils.widgets.dropdown(
    "profundidad", profundidad_previo, profundidad_opciones, "9-Profundidad"
)

## Feature engineering

La tabla ya tiene los tipos, sin `Permissions`, con los dos nombres de streaming acortados y con columnas válidas para Delta. Acá se aplican las reglas de filas.


In [ ]:
import pandas as pd

catalogo = spark.sql("SELECT current_catalog()").first()[0]
tabla = f"{catalogo}.bigdata_emanuel_rb.survey_cleaned"
pdf = spark.table(tabla).toPandas()

sin_etiqueta = pdf["music_effects"].isna() | (pdf["music_effects"].astype(str).str.strip() == "")
pdf = pdf.loc[~sin_etiqueta].copy()

tratar_bpm = dbutils.widgets.get("tratar_bpm")
bpm = pd.to_numeric(pdf["bpm"], errors="coerce")
if tratar_bpm == "anular_imposibles":
    pdf["bpm"] = bpm.where((bpm >= 40) & (bpm <= 250))
elif tratar_bpm == "conservar":
    pdf["bpm"] = bpm
else:
    raise ValueError(f"Tratar BPM desconocido: {tratar_bpm}")

print(f"tabla: {tabla}")
print(f"filas con etiqueta: {len(pdf)}")
print(pdf["music_effects"].value_counts().to_string())
print(f"tratar bpm: {tratar_bpm}")
print(f"bpm nulos: {int(pdf['bpm'].isna().sum())}")

## Matriz y corte

Hábitos de la tabla limpia. Los puntajes marcados en el widget se suman como columnas. El corte es 80/20 estratificado, `random_state=7`, igual para los cuatro.


In [ ]:
from sklearn.model_selection import train_test_split

PUNTAJES = ["anxiety", "depression", "insomnia", "ocd"]
NUMERICAS = ["age", "hours_per_day", "bpm"]
elegidos = [
    nombre.strip()
    for nombre in dbutils.widgets.get("scores").split(",")
    if nombre.strip() and nombre.strip() != "ninguno"
]
if "todos" in elegidos:
    elegidos = list(PUNTAJES)
desconocidos = [nombre for nombre in elegidos if nombre not in PUNTAJES]
if desconocidos:
    raise ValueError(f"Puntaje desconocido: {desconocidos}")

columnas = [nombre for nombre in pdf.columns if nombre not in ("timestamp", "music_effects", *PUNTAJES)]
columnas = columnas + [nombre for nombre in PUNTAJES if nombre in elegidos]

X = pdf[columnas]
y = pdf["music_effects"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)
print(f"columnas: {len(columnas)}")
print(f"puntajes: {elegidos or 'ninguno'}")
print(f"train: {len(X_train)}  test: {len(X_test)}")
print(y_test.value_counts().to_string())

## Modelos

Un pipeline por modelo: imputar, escalar las numéricas, one-hot de las categóricas, y el estimador. La imputación queda dentro del pipeline para que la mediana y la moda salgan del train.

Con `4-Como entrenar = automatico` esta celda registra exactamente esta configuración, aunque los otros widgets digan otra cosa. `C` de la logística no se aplica a los árboles. La profundidad `sin_limite` no se pasa: el bosque y LightGBM crecen sin tope, y CatBoost queda en la profundidad 6 de la librería.

| Run | Modelo | Balanceo |
|---|---|---|
| `exp_01` | Regresión logística, `C=1` | no |
| `exp_02` | Random forest, 100 árboles | `class_weight='balanced'` |
| `exp_03` | LightGBM, 100 árboles | `class_weight='balanced'` |
| `exp_04` | CatBoost, 100 iteraciones | `auto_class_weights='Balanced'` |

Con `4-Como entrenar = manual` se entrena solo el modelo del widget, con el balanceo, `C`, árboles y profundidad elegidos. El nombre por defecto es `manual-<modelo>`.

## Qué queda en MLflow

`mlflow.autolog()` queda prendido: al hacer `fit` anota los parámetros del pipeline, las métricas de ese ajuste y el modelo. Además se registran el F1 macro de train y de test, el F1 de cada clase y la matriz, porque `02_comparacion.ipynb` los busca por esos nombres y autolog no los usa.

El experimento `/Users/<usuario>/bigdata_emanuel_rb_tp_final` junta los entrenamientos. Cada uno es un run.

- **Parámetros:** modelo, modo, tratamiento del `bpm`, balanceo, `C`, árboles, profundidad, scores, filas y corte.
- **Métricas:** F1 macro de train, F1 macro de test, accuracy de test y F1 de `Improve`, `No effect` y `Worsen`.
- **Artefactos:** la matriz `matriz_confusion.png` y el pipeline, con la firma de las columnas de entrada.

In [ ]:
import matplotlib.pyplot as plt
import mlflow
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mlflow.autolog()
usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/bigdata_emanuel_rb_tp_final"
mlflow.set_experiment(experimento)

CLASES = ["Improve", "No effect", "Worsen"]
NOMBRES_COMPARACION = ["logistica", "random_forest", "lightgbm", "catboost"]

def preprocesar(columnas):
    numericas = [nombre for nombre in columnas if nombre in NUMERICAS or nombre in PUNTAJES]
    categoricas = [nombre for nombre in columnas if nombre not in numericas]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numericas),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categoricas),
    ])

def figura_matriz(y_real, y_pred, titulo):
    matriz = confusion_matrix(y_real, y_pred, labels=CLASES)
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    ax.imshow(matriz, cmap="Blues")
    for fila in range(matriz.shape[0]):
        for col in range(matriz.shape[1]):
            valor = matriz[fila, col]
            ax.text(col, fila, str(valor), ha="center", va="center",
                    color="white" if valor > matriz.max() / 2 else "black")
    ax.set_xticks(range(3), CLASES, rotation=15)
    ax.set_yticks(range(3), CLASES)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Realidad")
    ax.set_title(titulo, fontsize=10)
    fig.tight_layout()
    return fig

def estimador(nombre, balanceo, c_valor, n_arboles, profundidad):
    profundidad = None if profundidad == "sin_limite" else int(profundidad)
    n_arboles = int(n_arboles)
    c_valor = float(c_valor)
    pesar = balanceo == "balanced" or (
        balanceo == "como_la_comparacion" and nombre != "logistica"
    )
    if nombre == "logistica":
        kwargs = {"C": c_valor, "max_iter": 1000}
        if pesar:
            kwargs["class_weight"] = "balanced"
        return LogisticRegression(**kwargs)
    if nombre == "random_forest":
        kwargs = {"n_estimators": n_arboles, "random_state": 7, "n_jobs": -1}
        if profundidad is not None:
            kwargs["max_depth"] = profundidad
        if pesar:
            kwargs["class_weight"] = "balanced"
        return RandomForestClassifier(**kwargs)
    if nombre == "lightgbm":
        kwargs = {"n_estimators": n_arboles, "random_state": 7, "verbosity": -1}
        if profundidad is not None:
            kwargs["max_depth"] = profundidad
        if pesar:
            kwargs["class_weight"] = "balanced"
        return LGBMClassifier(**kwargs)
    if nombre == "catboost":
        kwargs = {
            "iterations": n_arboles,
            "random_state": 7,
            "verbose": 0,
            "allow_writing_files": False,
        }
        if profundidad is not None:
            kwargs["depth"] = profundidad
        if pesar:
            kwargs["auto_class_weights"] = "Balanced"
        return CatBoostClassifier(**kwargs)
    raise ValueError(f"Modelo desconocido: {nombre}")

que = dbutils.widgets.get("como_entrenar")
if que == "automatico":
    tanda = list(NOMBRES_COMPARACION)
    balanceo, c_valor, n_arboles, profundidad = "como_la_comparacion", "1", "100", "sin_limite"
    print("Modo automatico: hiperparámetros fijos de la comparación. No entran el nombre, el modelo, el balanceo, C, los árboles ni la profundidad.")
elif que == "manual":
    tanda = [dbutils.widgets.get("modelo")]
    balanceo = dbutils.widgets.get("balanceo")
    c_valor = dbutils.widgets.get("C")
    n_arboles = dbutils.widgets.get("n_arboles")
    profundidad = dbutils.widgets.get("profundidad")
else:
    raise ValueError(f"Como entrenar desconocido: {que}")

filas = []
figuras = []
for nombre in tanda:
    if que == "automatico":
        run_name = f"exp_{NOMBRES_COMPARACION.index(nombre) + 1:02d}"
    else:
        pedido = dbutils.widgets.get("nombre_experimento").strip()
        run_name = pedido or f"manual-{nombre}"
        if run_name in {f"exp_{i:02d}" for i in range(1, len(NOMBRES_COMPARACION) + 1)}:
            print(f"{run_name} es un nombre de la comparación. 02 va a tomar este run como ese experimento.")
    pipeline = Pipeline([
        ("preprocesamiento", preprocesar(list(X_train.columns))),
        ("modelo", estimador(nombre, balanceo, c_valor, n_arboles, profundidad)),
    ])
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "modelo": nombre,
            "que_entrenar": que,
            "tratar_bpm": tratar_bpm,
            "balanceo": balanceo,
            "C": c_valor,
            "n_arboles": n_arboles,
            "profundidad": profundidad,
            "puntajes_salud": ",".join(elegidos) if elegidos else "ninguno",
            "n_columnas": len(X_train.columns),
            "train_filas": len(X_train),
            "test_filas": len(X_test),
            "corte": "80/20 estratificado",
            "random_state": 7,
            "tabla": tabla,
        })
        pipeline.fit(X_train, y_train)
        pred_test = pipeline.predict(X_test)
        pred_train = pipeline.predict(X_train)
        f1_clases = f1_score(y_test, pred_test, average=None, labels=CLASES, zero_division=0)
        metricas = {
            "test_f1_macro": float(f1_score(y_test, pred_test, average="macro", zero_division=0)),
            "train_f1_macro": float(f1_score(y_train, pred_train, average="macro", zero_division=0)),
            "test_accuracy": float(accuracy_score(y_test, pred_test)),
            "test_f1_improve": float(f1_clases[0]),
            "test_f1_no_effect": float(f1_clases[1]),
            "test_f1_worsen": float(f1_clases[2]),
        }
        mlflow.log_metrics(metricas)
        fig = figura_matriz(y_test, pred_test, run_name)
        mlflow.log_figure(fig, "matriz_confusion.png")
        figuras.append(fig)
    filas.append({"run": run_name, "modelo": nombre, **metricas})
    print(f"Run '{run_name}' registrado")
    display(pd.DataFrame([metricas]).round(4))
    display(figuras[-1])
    plt.close(figuras[-1])

esta_tanda = pd.DataFrame(filas).sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
print(f"{len(esta_tanda)} runs de esta ejecución, del mejor al peor F1 macro de test")
display(esta_tanda.round(4))
print("experimento:", experimento)

## Runs de este experimento

Los de esta ejecución y los anteriores, del mejor al peor F1 macro de test. `02_comparacion.ipynb` arma la misma tabla con los gráficos.

In [ ]:
todos = mlflow.search_runs([mlflow.get_experiment_by_name(experimento).experiment_id])
todos = todos[todos["status"] == "FINISHED"].copy()
if "tags.mlflow.runName" in todos.columns:
    todos["run"] = todos["tags.mlflow.runName"]
else:
    todos["run"] = todos["run_name"]
columnas = {
    "run": "run",
    "params.modelo": "modelo",
    "params.que_entrenar": "que_entrenar",
    "params.tratar_bpm": "tratar_bpm",
    "params.balanceo": "balanceo",
    "params.C": "C",
    "params.n_arboles": "n_arboles",
    "params.profundidad": "profundidad",
    "params.puntajes_salud": "puntajes_salud",
    "metrics.train_f1_macro": "train_f1_macro",
    "metrics.test_f1_macro": "test_f1_macro",
    "metrics.test_accuracy": "test_accuracy",
    "metrics.test_f1_improve": "f1_improve",
    "metrics.test_f1_no_effect": "f1_no_effect",
    "metrics.test_f1_worsen": "f1_worsen",
}
for columna in columnas:
    if columna not in todos.columns:
        todos[columna] = None
tabla = todos.rename(columns=columnas)[list(columnas.values())]
tabla = tabla.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
print(f"Tenés {len(tabla)} runs terminados.")
display(tabla.round(4))